# Tutorial 02: Hospital Patient Cohort Ingestion & Inspection

This notebook ingests clinical data from `patients.csv`, computes summary
statistics, and visualises patient attributes.


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

Locates the `APDS/01_Hospital_Data_Setup/` project folder (anchored on `data/patients.csv`) by walking upward from the
current working directory — so the notebook runs identically from the repository
root, the project folder or its `notebooks/` sub-folder. On Google Colab the
repository is cloned into `/content/APSD_TAI_LAB` automatically (opening the
notebook directly from GitHub also works). The resolved project folder is added
to `sys.path` so its local modules (e.g. `utils`, `students`) import cleanly.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_NAME = "01_Hospital_Data_Setup"
ANCHOR_REL = "data/patients.csv"  # file that uniquely identifies the project folder
REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "APDS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
    PROJ_DIR = _repo / "APDS" / PROJECT_NAME
else:
    # Local IDE / Jupyter: find the project folder by walking upward from the
    # working directory (works from the repo root, the project folder or its
    # notebooks/ sub-folder, regardless of where Jupyter was launched).
    _cwd = Path.cwd().resolve()
    PROJ_DIR = next(
        (
            c / "APDS" / PROJECT_NAME
            for c in (_cwd, *_cwd.parents)
            if (c / "APDS" / PROJECT_NAME / ANCHOR_REL).exists()
        ),
        None,
    )
    if PROJ_DIR is None:
        raise FileNotFoundError(
            f"Project folder APDS/{PROJECT_NAME} (anchor: {ANCHOR_REL}) not found "
            f"above {_cwd}. Launch Jupyter from inside the cloned repository."
        )
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print(f"Project folder : {PROJ_DIR}")
print(f"Google Colab   : {IN_COLAB}")


### 1. Load the Patient Cohort
The CSV path is resolved from the project folder, not the working directory.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(PROJ_DIR / "data" / "patients.csv")
print(f"Cohort Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


### 2. Descriptive Summary Statistics
Review continuous metrics (Age, BloodPressure, Cholesterol) and frequency distributions across categorical features.


In [ ]:
print("--- Numerical Feature Summaries ---")
display(df.describe())

print("\n--- Sex Distribution ---")
print(df["Sex"].value_counts())

print("\n--- Clinical Diagnosis Distribution ---")
print(df["Diagnosis"].value_counts())


### 3. Clinical Visualization
Scatter plot of Patient Age vs. Serum Cholesterol, colored by Blood Pressure, followed by the `plt.plot` line view of Cholesterol vs. Age that TP2 (p. 2) explicitly requests.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

scatter = ax.scatter(
    df["Age"],
    df["Cholesterol"],
    c=df["BloodPressure"],
    cmap="coolwarm",
    s=20,
    alpha=0.5,
    edgecolors="black",
    linewidth=1.2,
)

cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Blood Pressure (mmHg)", fontsize=11)

ax.set_title("Patient Age vs. Serum Cholesterol Levels", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Patient Age (Years)", fontsize=11)
ax.set_ylabel("Serum Cholesterol (mg/dL)", fontsize=11)
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
# TP2 (p. 2): the requested plt.plot of Cholesterol as a function of
# Age — a line plot with markers. The colored scatter above is the
# bonus view of the same relationship (BloodPressure encoded as color).
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    df["Age"],
    df["Cholesterol"],
    marker="o",
    linestyle="-",
    color="tab:blue",
    markerfacecolor="white",
    markeredgecolor="tab:blue",
    alpha=0.8,
)

ax.set_title("Serum Cholesterol vs. Patient Age (TP2 — plt.plot)",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Patient Age (Years)", fontsize=11)
ax.set_ylabel("Serum Cholesterol (mg/dL)", fontsize=11)
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()
